# Phase 6.0-B: 跨物种保守性模式分析

**项目**: Human LncRNA Atlas

**分析目标**: 揭示 lncRNA 调控在进化中的保守性规律

**数据源**: `/api/v1/export/conservation` API

---

## 分析流程

1. 保守性数据获取
2. 保守性分类统计（2/3/4 物种保守）
3. 物种间共享矩阵
4. 保守性热力图
5. 进化距离与保守性相关性
6. 保守 vs 特异性 lncRNA 特征对比

In [ ]:
# 环境准备
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
from matplotlib_venn import venn4, venn3, venn2
import warnings

warnings.filterwarnings('ignore')
sns.set_palette("Set2")

API_BASE_URL = "http://localhost:8000/api/v1"
print("✅ 环境准备完成")

In [ ]:
# 获取保守性数据（2+ 物种保守）
response = requests.get(
    f"{API_BASE_URL}/export/conservation",
    params={"min_species_count": 2, "limit": 5000, "format": "json"}
)

data = response.json()
df_conservation = pd.DataFrame(data['data'])
print(f"✅ 获取了 {len(df_conservation)} 个保守 lncRNA")
df_conservation.head()

## 2. 保守性分类统计

In [ ]:
# 按物种保守数量分类
conservation_summary = df_conservation.groupby('species_count').agg({
    'core_id': 'count',
    'total_regulations': 'sum',
    'avg_binding_affinity': 'mean'
}).reset_index()

conservation_summary.columns = [
    'Species Count', 'lncRNA Count', 'Total Regulations', 'Avg BA'
]

# 计算占比
conservation_summary['lncRNA %'] = (
    100 * conservation_summary['lncRNA Count'] / conservation_summary['lncRNA Count'].sum()
)
conservation_summary['Regulations %'] = (
    100 * conservation_summary['Total Regulations'] / conservation_summary['Total Regulations'].sum()
)

print("=" * 80)
print("跨物种保守性统计")
print("=" * 80)
print(conservation_summary)

# 保存
conservation_summary.to_excel('results/conservation_summary.xlsx', index=False)
print("\n✅ 保守性统计已保存")

In [ ]:
# 保守性分布可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 饼图 - lncRNA 数量分布
axes[0].pie(
    conservation_summary['lncRNA Count'],
    labels=[f"{int(row['Species Count'])} Species\n({row['lncRNA Count']} lncRNAs)" 
            for _, row in conservation_summary.iterrows()],
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('pastel')
)
axes[0].set_title('lncRNA Conservation Distribution', fontsize=14, fontweight='bold')

# 柱状图 - 调控关系分布
axes[1].bar(
    conservation_summary['Species Count'].astype(str) + ' Species',
    conservation_summary['Total Regulations'],
    color=sns.color_palette('Set2'),
    edgecolor='black'
)
axes[1].set_xlabel('Conservation Level', fontsize=12)
axes[1].set_ylabel('Total Regulations', fontsize=12)
axes[1].set_title('Regulation Distribution by Conservation', fontsize=14, fontweight='bold')

# 添加数值标签
for i, row in conservation_summary.iterrows():
    axes[1].text(i, row['Total Regulations'], 
                f"{row['Total Regulations']:,.0f}",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/05_conservation_distribution.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存: figures/05_conservation_distribution.png")
plt.show()

## 3. 物种间共享矩阵与热力图

In [ ]:
# 构建物种间共享矩阵
# 注意：需要解析 lncrna_names 字段获取每个物种的具体基因名

# 物种名称映射
species_map = {
    'Human': 1,
    'Chimpanzee': 2,
    'Macaque': 3,
    'Marmoset': 4
}

# 提取每个 core_id 对应的物种集合
def extract_species_from_names(lncrna_names):
    """从 lncRNA 名称列表推断物种"""
    species_set = set()
    for name in lncrna_names:
        if '_chimp' in name:
            species_set.add('Chimpanzee')
        elif '_macaque' in name:
            species_set.add('Macaque')
        elif '_marmoset' in name:
            species_set.add('Marmoset')
        else:
            species_set.add('Human')
    return species_set

df_conservation['species_set'] = df_conservation['lncrna_names'].apply(extract_species_from_names)

# 构建 4x4 共享矩阵
species_names = ['Human', 'Chimpanzee', 'Macaque', 'Marmoset']
shared_matrix = np.zeros((4, 4), dtype=int)

for i, sp1 in enumerate(species_names):
    for j, sp2 in enumerate(species_names):
        if i == j:
            # 对角线：该物种的 lncRNA 总数
            shared_matrix[i, j] = sum(
                sp1 in species_set for species_set in df_conservation['species_set']
            )
        else:
            # 非对角线：两物种共享的 lncRNA 数
            shared_matrix[i, j] = sum(
                (sp1 in species_set and sp2 in species_set)
                for species_set in df_conservation['species_set']
            )

shared_df = pd.DataFrame(
    shared_matrix,
    index=species_names,
    columns=species_names
)

print("物种间共享 lncRNA 矩阵:")
print(shared_df)

In [ ]:
# 保守性热力图
plt.figure(figsize=(10, 8))

sns.heatmap(
    shared_df,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Shared lncRNA Count'},
    annot_kws={'fontsize': 12, 'fontweight': 'bold'}
)

plt.title('Cross-Species lncRNA Conservation Matrix', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Species', fontsize=13)
plt.ylabel('Species', fontsize=13)

plt.tight_layout()
plt.savefig('figures/06_conservation_heatmap.png', dpi=300, bbox_inches='tight')
print("✅ 热力图已保存: figures/06_conservation_heatmap.png")
plt.show()

## 4. 保守 lncRNA 特征分析

In [ ]:
# 比较不同保守等级的 lncRNA 特征
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 调控数量对比
df_conservation.boxplot(column='total_regulations', by='species_count', ax=axes[0])
axes[0].set_xlabel('Number of Conserved Species', fontsize=12)
axes[0].set_ylabel('Total Regulations', fontsize=12)
axes[0].set_title('Regulation Count by Conservation Level', fontsize=13, fontweight='bold')
plt.sca(axes[0])
plt.suptitle('')

# 平均 BA 对比
df_conservation.boxplot(column='avg_binding_affinity', by='species_count', ax=axes[1])
axes[1].set_xlabel('Number of Conserved Species', fontsize=12)
axes[1].set_ylabel('Average Binding Affinity', fontsize=12)
axes[1].set_title('BA by Conservation Level', fontsize=13, fontweight='bold')
plt.sca(axes[1])
plt.suptitle('')

# 散点图：物种数 vs 调控数量
scatter = axes[2].scatter(
    df_conservation['species_count'],
    df_conservation['total_regulations'],
    c=df_conservation['avg_binding_affinity'],
    s=100,
    cmap='viridis',
    alpha=0.6,
    edgecolors='black'
)
axes[2].set_xlabel('Species Count (Conservation)', fontsize=12)
axes[2].set_ylabel('Total Regulations', fontsize=12)
axes[2].set_title('Conservation vs Regulation Count', fontsize=13, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label('Avg BA', fontsize=11)

plt.tight_layout()
plt.savefig('figures/07_conservation_characteristics.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存: figures/07_conservation_characteristics.png")
plt.show()

## 5. Top 保守 lncRNA 排行榜

In [ ]:
# 筛选 4 物种保守的 lncRNA
highly_conserved = df_conservation[df_conservation['species_count'] == 4].copy()
highly_conserved = highly_conserved.sort_values('total_regulations', ascending=False)
highly_conserved['Rank'] = range(1, len(highly_conserved) + 1)

print(f"=" * 80)
print(f"4 物种保守 lncRNA 排行榜（共 {len(highly_conserved)} 个）")
print(f"=" * 80)
print(highly_conserved.head(20))

# 导出
highly_conserved.to_excel('results/highly_conserved_lncrnas_4species.xlsx', index=False)
print("\n✅ 保守 lncRNA 排行榜已保存")

## 6. Venn 图 - 物种间重叠

In [ ]:
# 构建 Venn 图数据
human_lncrnas = set(
    row['core_id'] for _, row in df_conservation.iterrows()
    if 'Human' in row['species_set']
)
chimp_lncrnas = set(
    row['core_id'] for _, row in df_conservation.iterrows()
    if 'Chimpanzee' in row['species_set']
)
macaque_lncrnas = set(
    row['core_id'] for _, row in df_conservation.iterrows()
    if 'Macaque' in row['species_set']
)
marmoset_lncrnas = set(
    row['core_id'] for _, row in df_conservation.iterrows()
    if 'Marmoset' in row['species_set']
)

print(f"物种 lncRNA 集合大小:")
print(f"  Human: {len(human_lncrnas)}")
print(f"  Chimpanzee: {len(chimp_lncrnas)}")
print(f"  Macaque: {len(macaque_lncrnas)}")
print(f"  Marmoset: {len(marmoset_lncrnas)}")

# 绘制 Venn 图（使用 matplotlib_venn）
# 注意：matplotlib_venn 最多支持 3 个集合的标准 Venn 图
# 对于 4 个集合，我们可以绘制多个 3-way Venn 图

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Venn 图 1: Human, Chimp, Macaque
venn3([human_lncrnas, chimp_lncrnas, macaque_lncrnas],
      set_labels=('Human', 'Chimpanzee', 'Macaque'),
      ax=axes[0])
axes[0].set_title('lncRNA Conservation: Human-Chimp-Macaque',
                 fontsize=14, fontweight='bold')

# Venn 图 2: Human, Chimp, Marmoset
venn3([human_lncrnas, chimp_lncrnas, marmoset_lncrnas],
      set_labels=('Human', 'Chimpanzee', 'Marmoset'),
      ax=axes[1])
axes[1].set_title('lncRNA Conservation: Human-Chimp-Marmoset',
                 fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/08_conservation_venn_diagrams.png', dpi=300, bbox_inches='tight')
print("✅ Venn 图已保存: figures/08_conservation_venn_diagrams.png")
plt.show()

## 7. 统计检验：保守性与调控强度

In [ ]:
# 假设检验：保守性更高的 lncRNA 是否调控更多靶基因？
print("=" * 80)
print("统计检验：保守性与调控强度的关系")
print("=" * 80)

# Kruskal-Wallis H 检验（非参数）
groups = [
    df_conservation[df_conservation['species_count'] == sc]['total_regulations'].values
    for sc in [2, 3, 4]
]
h_stat, p_value = stats.kruskal(*groups)

print(f"\nKruskal-Wallis H 检验:")
print(f"  H 统计量: {h_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  结论: {'保守性与调控强度有显著相关' if p_value < 0.05 else '无显著相关'} (α=0.05)")

# Spearman 相关性检验
corr, p_corr = stats.spearmanr(
    df_conservation['species_count'],
    df_conservation['total_regulations']
)

print(f"\nSpearman 相关性检验:")
print(f"  相关系数 (ρ): {corr:.4f}")
print(f"  p-value: {p_corr:.4e}")
print(f"  解释: {'正相关' if corr > 0 else '负相关'}，")
print(f"        {'统计显著' if p_corr < 0.05 else '不显著'} (α=0.05)")

# 保存统计结果
stats_results = pd.DataFrame([
    {'Test': 'Kruskal-Wallis H', 'Statistic': h_stat, 'p-value': p_value},
    {'Test': 'Spearman Correlation', 'Statistic': corr, 'p-value': p_corr}
])
stats_results.to_csv('results/conservation_statistical_tests.csv', index=False)
print("\n✅ 统计检验结果已保存")

## 8. 关键发现总结

In [ ]:
print("=" * 80)
print("跨物种保守性分析 - 关键发现")
print("=" * 80)

print(f"\n1. 保守性分层")
for _, row in conservation_summary.iterrows():
    print(f"   - {int(row['Species Count'])} 物种: {int(row['lncRNA Count'])} 个 lncRNA "
          f"({row['lncRNA %']:.1f}%), {int(row['Total Regulations'])} 调控关系")

print(f"\n2. 物种间共享模式")
print(f"   - Human-Chimp 共享: {shared_df.loc['Human', 'Chimpanzee']} 个 lncRNA（进化距离最近）")
print(f"   - Human-Marmoset 共享: {shared_df.loc['Human', 'Marmoset']} 个 lncRNA（进化距离最远）")

print(f"\n3. 统计显著性")
print(f"   - 保守性与调控强度: p = {p_value:.4e}")
print(f"   - Spearman 相关: ρ = {corr:.4f}, p = {p_corr:.4e}")

print(f"\n4. Top 保守 lncRNA")
top_1_conserved = highly_conserved.iloc[0]
print(f"   - Core ID: {top_1_conserved['core_id']}")
print(f"   - 调控关系: {top_1_conserved['total_regulations']}")
print(f"   - 平均 BA: {top_1_conserved['avg_binding_affinity']:.2f}")

print("\n" + "=" * 80)
print("分析完成！")
print("=" * 80)